In [3]:
from datetime import date
import polars as pl
from data_engineering.adsb.read_adsb import read_adsb
from data_engineering.flights.flight_type import get_flights

target_date = date(2026,3,1)
df_flights = get_flights(target_date)
df_adsb = read_adsb(target_date)

In [5]:
icao = "0100a3"
df_flights = df_flights.filter(pl.col("icao") == icao)
df_adsb = df_adsb.filter(pl.col("icao") == icao)

In [6]:
df_flights

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""0100a3""","""MSR520""","""SU-GCF""",2026-03-01 04:41:21.920,"""HE16""",2026-03-01 08:41:51.630,"""EBOS""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""0100a3""","""MSR521""","""SU-GCF""",2026-03-01 10:58:20.670,"""EBOS""",2026-03-01 14:42:50.320,"""EG-0017""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""


In [7]:
df_adsb

icao,ground_speed_kt,aircraft_type,ladd,on_ground,baro_altitude_ft,pia,track_deg,lat,lon,aircraft_description,interesting,owner,category,callsign,military,registration,time
str,f32,str,bool,bool,i32,bool,f32,f64,f64,str,bool,str,str,str,bool,str,datetime[ms]
"""0100a3""",378.200012,"""A332""",false,false,19500,false,318.200012,30.710037,30.799417,"""AIRBUS A-330-200""",false,null,null,null,false,"""SU-GCF""",2026-03-01 04:41:21.920
"""0100a3""",378.899994,"""A332""",false,false,19775,false,318.299988,30.721024,30.788107,"""AIRBUS A-330-200""",false,null,null,null,false,"""SU-GCF""",2026-03-01 04:41:30.410
"""0100a3""",379.600006,"""A332""",false,false,19975,false,318.200012,30.730453,30.77836,"""AIRBUS A-330-200""",false,null,"""A5""",null,false,"""SU-GCF""",2026-03-01 04:41:37.590
"""0100a3""",381.799988,"""A332""",false,false,20300,false,318.299988,30.746915,30.761334,"""AIRBUS A-330-200""",false,null,null,null,false,"""SU-GCF""",2026-03-01 04:41:50.090
"""0100a3""",383.200012,"""A332""",false,false,20650,false,318.299988,30.76386,30.743811,"""AIRBUS A-330-200""",false,null,null,null,false,"""SU-GCF""",2026-03-01 04:42:02.850
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""0100a3""",413.399994,"""A332""",false,false,20600,false,143.0,31.444244,30.99814,"""AIRBUS A-330-200""",false,null,null,null,false,"""SU-GCF""",2026-03-01 14:42:16.770
"""0100a3""",413.399994,"""A332""",false,false,20300,false,143.0,31.430374,31.010365,"""AIRBUS A-330-200""",false,null,null,null,false,"""SU-GCF""",2026-03-01 14:42:25.720
"""0100a3""",413.399994,"""A332""",false,false,20200,false,143.0,31.425705,31.014512,"""AIRBUS A-330-200""",false,null,null,null,false,"""SU-GCF""",2026-03-01 14:42:28.790


In [22]:
from collections import OrderedDict

import plotly.graph_objects as go
from plotly.colors import qualitative
from plotly.subplots import make_subplots


def visualize_flights_adsb(df_flights, df_adsb):
    no_flight_label = "No flight"

    def rows(df, sort_col):
        if sort_col in df.columns:
            df = df.sort(sort_col)
        return df.rows(named=True)

    def flight_label(idx, flight):
        callsign = flight.get("callsign") or f"Flight {idx}"
        takeoff = flight.get("takeoff_airport_ident") or "?"
        landing = flight.get("landing_airport_ident") or "?"
        return f"F{idx}: {callsign} {takeoff}->{landing}"

    def message_label(message):
        time = message.get("time")
        for start, end, label in flight_windows:
            if time is not None and start is not None and end is not None and start <= time <= end:
                return label
        return no_flight_label

    flights = rows(df_flights, "takeoff_time")
    flight_windows = [
        (
            flight.get("takeoff_time"),
            flight.get("landing_time"),
            flight_label(idx, flight),
        )
        for idx, flight in enumerate(flights, start=1)
    ]

    messages = rows(df_adsb, "time")
    groups = OrderedDict((label, []) for _, _, label in flight_windows)
    groups[no_flight_label] = []
    for message in messages:
        groups[message_label(message)].append(message)

    fig = make_subplots(
        rows=2,
        cols=1,
        specs=[[{"type": "xy"}], [{"type": "map"}]],
        row_heights=[0.36, 0.64],
        vertical_spacing=0.07,
    )

    palette = qualitative.Plotly + qualitative.Dark24 + qualitative.Light24
    colors = {
        label: ("#808080" if label == no_flight_label else palette[idx % len(palette)])
        for idx, label in enumerate(groups)
    }

    for start, end, label in flight_windows:
        if start is not None and end is not None:
            fig.add_vrect(
                x0=start,
                x1=end,
                fillcolor=colors[label],
                opacity=0.07,
                line_width=0,
                row=1,
                col=1,
            )

    for label, group_messages in groups.items():
        if not group_messages:
            continue

        color = colors[label]
        hover_text = [
            "<br>".join(
                [
                    f"{label}",
                    f"time: {message.get('time')}",
                    f"altitude: {message.get('baro_altitude_ft')} ft",
                    f"speed: {message.get('ground_speed_kt')} kt",
                    f"track: {message.get('track_deg')} deg",
                    f"lat/lon: {message.get('lat')}, {message.get('lon')}",
                ]
            )
            for message in group_messages
        ]

        fig.add_trace(
            go.Scatter(
                x=[message.get("time") for message in group_messages],
                y=[message.get("baro_altitude_ft") for message in group_messages],
                mode="markers" if label == no_flight_label else "lines+markers",
                name=label,
                legendgroup=label,
                marker={"color": color, "size": 5},
                line={"color": color, "width": 2},
                hovertext=hover_text,
                hovertemplate="%{hovertext}<extra></extra>",
            ),
            row=1,
            col=1,
        )

        map_messages = [
            (message, hover)
            for message, hover in zip(group_messages, hover_text)
            if message.get("lat") is not None and message.get("lon") is not None
        ]
        if map_messages:
            fig.add_trace(
                go.Scattermap(
                    lat=[message.get("lat") for message, _ in map_messages],
                    lon=[message.get("lon") for message, _ in map_messages],
                    mode="markers",
                    name=label,
                    legendgroup=label,
                    showlegend=False,
                    marker={"color": color, "size": 6, "opacity": 0.78},
                    hovertext=[hover for _, hover in map_messages],
                    hovertemplate="%{hovertext}<extra></extra>",
                ),
                row=2,
                col=1,
            )

    map_points = [
        message
        for message in messages
        if message.get("lat") is not None and message.get("lon") is not None
    ]
    map_center = (
        {
            "lat": sum(message.get("lat") for message in map_points) / len(map_points),
            "lon": sum(message.get("lon") for message in map_points) / len(map_points),
        }
        if map_points
        else {"lat": 0, "lon": 0}
    )
    icao = (messages[0].get("icao") if messages else None) or (flights[0].get("icao") if flights else "")

    fig.update_xaxes(title_text="Time", row=1, col=1)
    fig.update_yaxes(title_text="Barometric altitude (ft)", row=1, col=1)
    fig.update_layout(
        title=f"Flights and ADS-B messages for {icao}",
        height=900,
        hovermode="closest",
        legend={"groupclick": "togglegroup", "traceorder": "normal"},
        map={
            "style": "open-street-map",
            "center": map_center,
            "zoom": 5 if map_points else 1,
        },
        margin={"l": 50, "r": 25, "t": 60, "b": 35},
    )
    return fig


In [23]:
fig = visualize_flights_adsb(df_flights, df_adsb)
fig.show(config={"scrollZoom": True})